In [1]:
import wave
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy import signal
import numpy as np
import scipy.io.wavfile as wavfile
import pygame


pygame 2.3.0 (SDL 2.24.2, Python 3.9.16)
Hello from the pygame community. https://www.pygame.org/contribute.html


Plotting

Waveform

In [2]:
if not os.path.exists('waveforms'):
    os.mkdir('waveforms')

for i in range(1, 9):
    filename = f"rec{i}.wav"
    
    with wave.open(filename, 'r') as wav_file:
        # Extrahieren von Informationen aus der Wave-Datei
        frames = wav_file.readframes(-1)
        sample_rate = wav_file.getframerate()
        num_channels = wav_file.getnchannels()
        sample_width = wav_file.getsampwidth()



    # Umwandeln von Byte-Daten in NumPy-Array
    frames = np.frombuffer(frames, dtype=np.int16)

    # Normalisieren der Daten auf den Bereich [-1, 1]
    frames = frames / 2**(8*sample_width-1)

    # Erstellen der x-Achse
    time = np.arange(0, len(frames)) / sample_rate
    print(f"{filename} - sample_rate: {sample_rate} - num_channels: {num_channels} - sample_width: {sample_width}")
    # Plotten der Waveform
    # plt.figure()
    # plt.plot(time, frames)
    # plt.xlabel('Time (s)')
    # plt.ylabel('Amplitude')
    # plt.title(f"Waveform {i}")
    # plt.savefig(f"waveforms/waveform_{i}.png")
    # plt.show()


rec1.wav - sample_rate: 16000 - num_channels: 1 - sample_width: 2
rec2.wav - sample_rate: 16000 - num_channels: 1 - sample_width: 2
rec3.wav - sample_rate: 16000 - num_channels: 1 - sample_width: 2
rec4.wav - sample_rate: 16000 - num_channels: 1 - sample_width: 2
rec5.wav - sample_rate: 16000 - num_channels: 1 - sample_width: 2
rec6.wav - sample_rate: 16000 - num_channels: 1 - sample_width: 2
rec7.wav - sample_rate: 16000 - num_channels: 1 - sample_width: 2
rec8.wav - sample_rate: 16000 - num_channels: 1 - sample_width: 2


Fourier-Transformation (FFT)

In [3]:
if not os.path.exists('fourier'):
    os.mkdir('fourier')

for i in range(1, 9):
    filename = f"rec{i}.wav"
    
    with wave.open(filename, 'r') as wav_file:
        # Extrahieren von Informationen aus der Wave-Datei
        frames = wav_file.readframes(-1)
        sample_rate = wav_file.getframerate()
        num_channels = wav_file.getnchannels()
        sample_width = wav_file.getsampwidth()

    # Umwandeln von Byte-Daten in NumPy-Array
    frames = np.frombuffer(frames, dtype=np.int16)

    # Normalisieren der Daten auf den Bereich [-1, 1]
    frames = frames / 2**(8*sample_width-1)

    # Berechnen der Fourier-Transformation
    freq = np.fft.rfftfreq(len(frames), d=1/sample_rate)
    freq_amp = np.abs(np.fft.rfft(frames))
    print(f"{filename} - sample_rate: {sample_rate} - num_channels: {num_channels} - sample_width: {sample_width}")

    # Plotten der Frequenzamplitude
    # plt.figure()
    # plt.plot(freq, freq_amp)
    # plt.xlabel('Frequency (Hz)')
    # plt.ylabel('Amplitude')
    # plt.title(f"Frequency Amplitude {i}")
    # plt.savefig(f"fourier/fourier_{i}.png")
    # plt.show()


rec1.wav - sample_rate: 16000 - num_channels: 1 - sample_width: 2
rec2.wav - sample_rate: 16000 - num_channels: 1 - sample_width: 2
rec3.wav - sample_rate: 16000 - num_channels: 1 - sample_width: 2
rec4.wav - sample_rate: 16000 - num_channels: 1 - sample_width: 2
rec5.wav - sample_rate: 16000 - num_channels: 1 - sample_width: 2
rec6.wav - sample_rate: 16000 - num_channels: 1 - sample_width: 2
rec7.wav - sample_rate: 16000 - num_channels: 1 - sample_width: 2
rec8.wav - sample_rate: 16000 - num_channels: 1 - sample_width: 2


Processing

Remove not needed frequencies

In [4]:
# Laden des Audiosignals
fs, audio_signal = wavfile.read('rec2.wav')
low, high = 100, 1000
# Einstellungen für die FFT
nperseg = max(1024, 2 * int(fs / (high - low)))  # Länge jedes FFT-Segments
noverlap = int(nperseg / 2)  # Überlappung zwischen den Segmenten

# Berechnung der FFT auf jedem Segment
f, t, Zxx = signal.stft(audio_signal, fs, nperseg=nperseg, noverlap=noverlap)

# Extraktion des gewünschten Frequenzbereichs (z.B. von 100 Hz bis 1000 Hz)

band = np.logical_and(f >= low, f <= high)
relevant_Zxx = Zxx[band, :]

# Rücktransformation in den Zeitbereich
_, reconstructed_signal = signal.istft(
    relevant_Zxx, fs, window='hann', nperseg=nperseg, noverlap=noverlap)

# Umwandlung des Signals in das Integer-Format
reconstructed_signal = np.int16(
    np.round(reconstructed_signal / np.max(np.abs(reconstructed_signal)) * 32767))

# Schreiben des Signals in eine WAV-Datei
wavfile.write('relevant_audio_file.wav', fs, reconstructed_signal)


ValueError: operands could not be broadcast together with shapes (114,) (1024,) 

: 